In [ ]:
import scanpy as sc
import matplotlib.pyplot as plt

plt.style.use('default')

In [ ]:
PHH_pred_ad = sc.read_h5ad('../data/LUAD/ACH-000587_Osimertinib_12328pred.h5ad')
print(PHH_pred_ad.obs.shape)

In [ ]:
sc.pp.highly_variable_genes(PHH_pred_ad, n_top_genes=2000)
adata = PHH_pred_ad[:, PHH_pred_ad.var.highly_variable]

In [ ]:
sc.pp.scale(adata)
sc.tl.pca(adata, svd_solver='arpack')

sc.pp.neighbors(adata)
sc.tl.leiden(adata, resolution=1)
sc.tl.umap(adata)
sc.pl.umap(adata, color='leiden',show=False)

sc.pl.umap(adata, color=['pert_itime'],cmap='coolwarm',show=False)


sc.tl.rank_genes_groups(adata, 'leiden')
sc.pl.rank_genes_groups(adata)

In [ ]:

sc.tl.rank_genes_groups(
    adata,
    groupby='leiden',
    groups=['0'],
    reference='4',
    method='wilcoxon',
    key_added="rank_genes_0_vs_4"
)

sc.pl.rank_genes_groups(
    adata,
    key="rank_genes_0_vs_4",
    groups=['0']
)

In [ ]:

top25_genes = sc.get.rank_genes_groups_df(adata, key="rank_genes_4_vs_0", group='4')['names'][:25].tolist()
print("Top 20 Up Regulated Genes:", top25_genes)
# target_genes = ['7474','60681','4982','5270','11211','1031']
new_order = ['4','2','3','1','6','0','5']
adata.obs['leiden'] = adata.obs['leiden'].cat.reorder_categories(new_order)

# # 1. Violin plot: The gene expression distribution in the two clusters can be clearly seen
# sc.pl.violin(
#     adata,
#     keys=top5_genes,
#     groupby='leiden',
#     palette='coolwarm', 
#     use_raw=False,
#     # order=['0','2','1','3','4']
# )


# # 2. Dot plot: can simultaneously display the expression proportion and average expression level of multiple genes in two clusters
# sc.pl.dotplot(
#     adata,
#     var_names=top5_genes,
#     groupby='leiden',
#     use_raw=False,
#     standard_scale='var',
# )


# 3. heatmap：clearly see gene expression patterns across different clusters.
sc.pl.heatmap(
    adata,
    var_names=top25_genes,
    groupby='leiden',
    cmap='coolwarm',
    use_raw=False,
    # show=False,
    # dendrogram=True,  # cluster the genes
    # save='_leiden_upheatmap.svg' # save as svg file (optional)
)

In [ ]:
top25_genes = sc.get.rank_genes_groups_df(adata, key="rank_genes_0_vs_4", group='0')['names'][:25].tolist()
print("Top 20 Down Regulated Genes:", top25_genes)
# target_genes = ['7474','60681','4982','5270','11211','1031']
new_order = ['4','2','3','1','6','0','5'] # the order you want to display the clusters in the heatmap
adata.obs['leiden'] = adata.obs['leiden'].cat.reorder_categories(new_order)


sc.pl.heatmap(
    adata,
    var_names=top25_genes, 
    groupby='leiden',     
    cmap='coolwarm',       
    use_raw=False,        
    # show=False,           
    # dendrogram=True,      
    # save='_leiden_downheatmap.svg'
)